# DELCODE Duplicate Subject Audit

This notebook scans subject folders under `__v0__/fmri/<category>/sub-*` and reports duplicates across categories.

In [1]:
from pathlib import Path
from collections import defaultdict
import csv

try:
    import pandas as pd
except ImportError:
    pd = None

SOURCE_FMRI = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v0__/fmri')
OUTPUT_DIR = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v1__')

if not SOURCE_FMRI.exists():
    raise FileNotFoundError(f'Source path not found: {SOURCE_FMRI}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Source: {SOURCE_FMRI}')
print(f'Output: {OUTPUT_DIR}')

Source: /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v0__/fmri
Output: /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v1__


In [2]:
# Build appearance records: one record for each category/sub-* occurrence
appearances = []
subject_to_categories = defaultdict(set)
subject_to_paths = defaultdict(list)

category_dirs = sorted([p for p in SOURCE_FMRI.iterdir() if p.is_dir()])

for category_dir in category_dirs:
    category = category_dir.name
    for subject_dir in sorted(category_dir.glob('sub-*')):
        if not subject_dir.is_dir():
            continue
        subject_id = subject_dir.name
        rec = {
            'subject_id': subject_id,
            'category': category,
            'source_path': str(subject_dir)
        }
        appearances.append(rec)
        subject_to_categories[subject_id].add(category)
        subject_to_paths[subject_id].append(str(subject_dir))

print(f'Categories scanned: {len(category_dirs)}')
print(f'Total subject appearances: {len(appearances)}')
print(f'Unique subject IDs: {len(subject_to_categories)}')

Categories scanned: 4
Total subject appearances: 898
Unique subject IDs: 841


In [3]:
# Compute duplicate summary (subjects present in more than one category)
duplicate_subjects = []
for subject_id in sorted(subject_to_categories.keys()):
    categories = sorted(subject_to_categories[subject_id])
    paths = sorted(subject_to_paths[subject_id])
    if len(categories) > 1:
        duplicate_subjects.append({
            'subject_id': subject_id,
            'n_categories': len(categories),
            'categories': '; '.join(categories),
            'source_paths': '; '.join(paths)
        })

summary_rows = [
    {'metric': 'total_appearances', 'value': len(appearances)},
    {'metric': 'unique_subject_ids', 'value': len(subject_to_categories)},
    {'metric': 'duplicate_subject_ids', 'value': len(duplicate_subjects)}
]

print('Summary:')
for row in summary_rows:
    print(f"- {row['metric']}: {row['value']}")

Summary:
- total_appearances: 898
- unique_subject_ids: 841
- duplicate_subject_ids: 57


In [12]:
df = pd.DataFrame(duplicate_subjects)
df.to_csv('duplicates_report.csv', index=False)

In [13]:
# Verify each subject directory under __v0__/fmri/* contains only ses-01 (if any ses-* dirs exist)
from collections import Counter

subject_session_report = []
only_ses01_count = 0
has_other_sessions_count = 0
no_session_dir_count = 0

for category_dir in sorted([p for p in SOURCE_FMRI.iterdir() if p.is_dir()]):
    category = category_dir.name
    for subject_dir in sorted(category_dir.glob('sub-*')):
        if not subject_dir.is_dir():
            continue

        ses_dirs = sorted([p.name for p in subject_dir.iterdir() if p.is_dir() and p.name.startswith('ses-')])

        if not ses_dirs:
            status = 'no_ses_dir_found'
            no_session_dir_count += 1
        elif set(ses_dirs) == {'ses-01'}:
            status = 'only_ses-01'
            only_ses01_count += 1
        else:
            status = 'has_non_ses-01_or_multiple_sessions'
            has_other_sessions_count += 1

        subject_session_report.append({
            'category': category,
            'subject_id': subject_dir.name,
            'status': status,
            'sessions_found': '; '.join(ses_dirs) if ses_dirs else ''
        })

print('Session structure summary:')
print(f"- total subject appearances checked: {len(subject_session_report)}")
print(f"- only ses-01: {only_ses01_count}")
print(f"- has non-ses-01 or multiple sessions: {has_other_sessions_count}")
print(f"- no ses-* directory found: {no_session_dir_count}")

if pd is not None:
    session_df = pd.DataFrame(subject_session_report)
    issues_df = session_df[session_df['status'] != 'only_ses-01'].copy()
    print('')
    print(f"Subjects not matching 'only ses-01': {len(issues_df)}")
    display(issues_df.sort_values(['status', 'category', 'subject_id']).head(100))

    session_csv = OUTPUT_DIR / 'subject_session_check.csv'
    issues_csv = OUTPUT_DIR / 'subject_session_check_issues.csv'
    session_df.sort_values(['category', 'subject_id']).to_csv(session_csv, index=False)
    issues_df.sort_values(['status', 'category', 'subject_id']).to_csv(issues_csv, index=False)
    print(f"\nWrote: {session_csv}")
    print(f"Wrote: {issues_csv}")
else:
    issues = [r for r in subject_session_report if r['status'] != 'only_ses-01']
    print('')
    print(f"Subjects not matching 'only ses-01': {len(issues)}")
    for row in issues[:100]:
        print(f"{row['category']} | {row['subject_id']} | {row['status']} | {row['sessions_found']}")

Session structure summary:
- total subject appearances checked: 898
- only ses-01: 898
- has non-ses-01 or multiple sessions: 0
- no ses-* directory found: 0

Subjects not matching 'only ses-01': 0


,category,subject_id,status,sessions_found



Wrote: /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v1__/subject_session_check.csv
Wrote: /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v1__/subject_session_check_issues.csv


In [14]:
# Compare two specific NIfTI files to confirm whether they are exact duplicates
from pathlib import Path
import hashlib

file_a = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v0__/fmri/AD_postprocessed_v2/sub-01dc83c85/ses-01/sub-01dc83c85_ses-01_task-rest_space-MNI152NLin2009cAsym_res-2_desc-ICAAROMA2Phys1GS_bold_reoriented.nii.gz')
file_b = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v0__/fmri/Converter_postprocessed_v2/sub-01dc83c85/ses-01/sub-01dc83c85_ses-01_task-rest_space-MNI152NLin2009cAsym_res-2_desc-ICAAROMA2Phys1GS_bold_reoriented.nii.gz')

def sha256_of_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

if not file_a.exists() or not file_b.exists():
    print('One or both files do not exist:')
    print(f'- exists A: {file_a.exists()} | {file_a}')
    print(f'- exists B: {file_b.exists()} | {file_b}')
else:
    size_a = file_a.stat().st_size
    size_b = file_b.stat().st_size
    print(f'Size A: {size_a} bytes')
    print(f'Size B: {size_b} bytes')
    print(f'Same size: {size_a == size_b}')

    hash_a = sha256_of_file(file_a)
    hash_b = sha256_of_file(file_b)
    print(f'SHA256 A: {hash_a}')
    print(f'SHA256 B: {hash_b}')

    is_exact_duplicate = (size_a == size_b) and (hash_a == hash_b)
    print(f'Exact duplicate (byte-identical): {is_exact_duplicate}')

Size A: 335377632 bytes
Size B: 337386784 bytes
Same size: False
SHA256 A: b4f430b304705009989f32723ccb8da408b515c083c96efecd342a00c76006e5
SHA256 B: 2da63cb2b34a2f51426acd01a8ad18af62b6536c7a820c3b7b722319788e23e9
Exact duplicate (byte-identical): False


In [ ]:
# Search for matching DICOM files and compare their metadata when raw scans are available
from pathlib import Path
from collections import Counter
import hashlib

try:
    import pydicom
except ImportError:
    pydicom = None

subject_id = 'sub-01dc83c85'
category_a = 'AD_postprocessed_v2'
category_b = 'Converter_postprocessed_v2'

# Adjust these roots if your raw DICOMs live elsewhere.
dicom_search_roots = [
    Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE'),
]

def looks_like_dicom(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in {'.dcm', '.ima', ''}

def candidate_dicom_dirs(root: Path, subject_name: str):
    candidates = []
    if not root.exists():
        return candidates
    for path in root.rglob(f'*{subject_name}*'):
        if path.is_dir():
            candidates.append(path)
    return sorted(set(candidates))

def collect_dicom_files(subject_name: str, preferred_token: str):
    dicom_files = []
    checked_dirs = []
    for root in dicom_search_roots:
        for directory in candidate_dicom_dirs(root, subject_name):
            checked_dirs.append(directory)
            if preferred_token not in str(directory):
                continue
            files = [p for p in directory.rglob('*') if looks_like_dicom(p)]
            if files:
                dicom_files.extend(files)
    return sorted(set(dicom_files)), checked_dirs

def safe_get(dataset, keyword):
    value = getattr(dataset, keyword, None)
    if value is None:
        return None
    return str(value)

def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def summarize_dicom_metadata(files):
    rows = []
    for path in files:
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
        except Exception as exc:
            rows.append({
                'path': str(path),
                'read_error': str(exc),
            })
            continue
        rows.append({
            'path': str(path),
            'PatientID': safe_get(ds, 'PatientID'),
            'PatientName': safe_get(ds, 'PatientName'),
            'StudyInstanceUID': safe_get(ds, 'StudyInstanceUID'),
            'SeriesInstanceUID': safe_get(ds, 'SeriesInstanceUID'),
            'SOPInstanceUID': safe_get(ds, 'SOPInstanceUID'),
            'StudyDate': safe_get(ds, 'StudyDate'),
            'SeriesDate': safe_get(ds, 'SeriesDate'),
            'AcquisitionDate': safe_get(ds, 'AcquisitionDate'),
            'Modality': safe_get(ds, 'Modality'),
            'SeriesDescription': safe_get(ds, 'SeriesDescription'),
            'ProtocolName': safe_get(ds, 'ProtocolName'),
            'Manufacturer': safe_get(ds, 'Manufacturer'),
            'ManufacturerModelName': safe_get(ds, 'ManufacturerModelName'),
            'MagneticFieldStrength': safe_get(ds, 'MagneticFieldStrength'),
            'RepetitionTime': safe_get(ds, 'RepetitionTime'),
            'EchoTime': safe_get(ds, 'EchoTime'),
            'FlipAngle': safe_get(ds, 'FlipAngle'),
            'Rows': safe_get(ds, 'Rows'),
            'Columns': safe_get(ds, 'Columns'),
            'PixelSpacing': safe_get(ds, 'PixelSpacing'),
            'SliceThickness': safe_get(ds, 'SliceThickness'),
            'sha256': file_sha256(path),
        })
    return rows

files_a, checked_dirs_a = collect_dicom_files(subject_id, category_a)
files_b, checked_dirs_b = collect_dicom_files(subject_id, category_b)

print(f'Searched roots: {[str(p) for p in dicom_search_roots]}')
print(f'{category_a} candidate DICOM files found: {len(files_a)}')
print(f'{category_b} candidate DICOM files found: {len(files_b)}')

if pydicom is None:
    print('pydicom is not installed, so DICOM metadata cannot be read in this notebook yet.')
elif not files_a or not files_b:
    print('Raw DICOM files were not found for one or both categories in the current repository/search roots.')
    print('If you have raw scan folders elsewhere, update dicom_search_roots in this cell and rerun it.')
    print('Sample checked directories:')
    for path in (checked_dirs_a + checked_dirs_b)[:20]:
        print(f'- {path}')
else:
    meta_a = summarize_dicom_metadata(files_a)
    meta_b = summarize_dicom_metadata(files_b)

    if pd is not None:
        df_a = pd.DataFrame(meta_a)
        df_b = pd.DataFrame(meta_b)

        compare_cols = [
            'PatientID', 'PatientName', 'StudyInstanceUID', 'SeriesInstanceUID', 'StudyDate',
            'SeriesDate', 'AcquisitionDate', 'Modality', 'SeriesDescription', 'ProtocolName',
            'Manufacturer', 'ManufacturerModelName', 'MagneticFieldStrength', 'RepetitionTime',
            'EchoTime', 'FlipAngle', 'Rows', 'Columns', 'PixelSpacing', 'SliceThickness'
        ]

        summary = []
        for col in compare_cols:
            values_a = Counter(v for v in df_a.get(col, []).tolist() if v not in [None, ''])
            values_b = Counter(v for v in df_b.get(col, []).tolist() if v not in [None, ''])
            summary.append({
                'field': col,
                'unique_values_in_a': '; '.join(sorted(map(str, values_a.keys()))[:10]),
                'unique_values_in_b': '; '.join(sorted(map(str, values_b.keys()))[:10]),
                'same_unique_values': set(values_a.keys()) == set(values_b.keys()),
            })

        summary_df = pd.DataFrame(summary)
        print('Field-level metadata comparison:')
        display(summary_df)

        out_a = OUTPUT_DIR / f'{subject_id}_{category_a}_dicom_metadata.csv'
        out_b = OUTPUT_DIR / f'{subject_id}_{category_b}_dicom_metadata.csv'
        out_cmp = OUTPUT_DIR / f'{subject_id}_{category_a}_vs_{category_b}_dicom_metadata_comparison.csv'
        df_a.to_csv(out_a, index=False)
        df_b.to_csv(out_b, index=False)
        summary_df.to_csv(out_cmp, index=False)
        print(f'\nWrote: {out_a}')
        print(f'Wrote: {out_b}')
        print(f'Wrote: {out_cmp}')
    else:
        print('pandas is not installed, so only file counts are shown.')
        print(f'{category_a} files: {len(meta_a)}')
        print(f'{category_b} files: {len(meta_b)}')